<a href="https://colab.research.google.com/github/Ishan-debugg/Fine-Tuning-LoRA/blob/main/notebooks/phase2sft.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
print(torch.cuda.get_device_name(0))
# Should print: Tesla T4

Tesla T4


In [2]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install trl==0.8.6 wandb datasets

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-t1wd7qvz/unsloth_a0e8c0f5eb6d4fef90fcec6a802d9c56
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-t1wd7qvz/unsloth_a0e8c0f5eb6d4fef90fcec6a802d9c56
  Resolved https://github.com/unslothai/unsloth.git to commit 2819e29474a58d69489ef5ce235cce6be9a2c8cd
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached trl-0.24.0-py3-none-any.whl.metadata (11 kB)
Using cached trl-0.24.0-py3-none-any.whl (423 kB)
  Attempting uninstall: trl
    Found existing installation: trl 0.8.6
    Uninstalling trl-0.8.6:
      Successfully uninstalled trl-0.8.6
  Using cached trl-0.8.6-py3-none-any.whl.metadata (11 kB)
Using cached trl-0.8.6-py3-none-any.whl (245 kB)
  Attempting uninstall: trl
    Found existing installation: trl 0.24.0
    Uninstalling trl-0.24.0:
      Successfully 

In [3]:
from google.colab import drive
drive.mount('/content/drive')

from datasets import load_from_disk
dataset = load_from_disk("/content/drive/MyDrive/fine-tuning-project/curated_data")
print(f"Train: {len(dataset['train']):,}")
print(f"Val:   {len(dataset['validation']):,}")

Mounted at /content/drive
Train: 2,500
Val:   2,732


In [4]:
# Cell 2: All imports and config in one place
# Read every config value and understand why it exists

import os
import json
import random
import torch
import numpy as np
from pathlib import Path
from datetime import datetime
from collections import defaultdict

from datasets import load_from_disk, Dataset
from transformers import TrainingArguments, AutoTokenizer
from trl import SFTTrainer, DataCollatorForCompletionOnlyLM
from unsloth import FastLanguageModel
import wandb

# ── Reproducibility ────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── Model Config ───────────────────────────────────────────────────────────────
MODEL_CONFIG = {
    # The model you're fine-tuning
    "base_model": "Qwen/Qwen2.5-1.5B-Instruct",

    # 4-bit quantization: loads frozen weights in NF4 (NormalFloat4) format
    # This is what makes QLoRA possible on a single GPU
    # Without this, 1.5B model needs ~3GB; with 4-bit, it needs ~1GB
    "load_in_4bit": True,

    # Maximum sequence length — affects VRAM usage quadratically via attention
    # 2048 is safe for T4 with batch_size=4
    # Increase to 4096 on A100 if your examples are long
    "max_seq_length": 2048,

    # Data type for computation (not storage)
    # None = auto-detect (bfloat16 on A100, float16 on T4)
    "dtype": None,
}

# ── LoRA Config ────────────────────────────────────────────────────────────────
# You will understand each of these after Cell 7's explanation
LORA_CONFIG = {
    "r": 16,                    # Rank of the update matrices
    "lora_alpha": 32,           # Scale factor: effective_lr = lr * (alpha/r) = lr * 2
    "lora_dropout": 0.05,       # Dropout on LoRA layers (regularization)
    "bias": "none",             # Don't train bias terms (memory saving)
    "target_modules": [         # Which weight matrices to add LoRA to
        "q_proj",               # Query projection in attention
        "k_proj",               # Key projection in attention
        "v_proj",               # Value projection in attention
        "o_proj",               # Output projection in attention
        "gate_proj",            # MLP gate (SwiGLU architecture)
        "up_proj",              # MLP up projection
        "down_proj",            # MLP down projection
    ],
    "use_gradient_checkpointing": "unsloth",  # Memory saving: recompute activations
    "random_state": SEED,
    "use_rslora": False,        # Rank-stabilized LoRA — try True in iteration 2
}

# ── Training Config ────────────────────────────────────────────────────────────
TRAIN_CONFIG = {
    "num_train_epochs": 3,

    # Batch size per GPU step
    # T4 (16GB): use 4 | A100 (40GB): use 8
    "per_device_train_batch_size": 4,

    # Gradient accumulation: effective batch = batch_size × accumulation_steps
    # With batch=4 and accum=4: effective batch = 16
    # WHY: Large effective batch = more stable gradients
    # Simulates training with 16 examples per update without needing 16×VRAM
    "gradient_accumulation_steps": 4,

    # Learning rate — the most sensitive hyperparameter
    # 2e-4 is standard for LoRA SFT; too high → loss explosion, too low → no learning
    "learning_rate": 2e-4,

    # Warmup: linearly increase LR from 0 to 2e-4 over first 3% of steps
    # WHY: Prevents the model from making wild updates in the first few batches
    # before it's "calibrated" to the new task
    "warmup_ratio": 0.03,

    # Cosine schedule: LR decays from 2e-4 to ~0 following a cosine curve
    # Better than linear for fine-tuning — smooth landing
    "lr_scheduler_type": "cosine",

    # Mixed precision training — uses float16 for forward pass, float32 for updates
    # Cuts VRAM usage by ~40% with minimal accuracy impact
    "fp16": not torch.cuda.is_bf16_supported(),
    "bf16": torch.cuda.is_bf16_supported(),  # Use bfloat16 on A100 (better range)

    # Logging: record metrics every 10 steps to WandB
    "logging_steps": 10,

    # Evaluate on validation set every 100 steps
    "eval_steps": 100,
    "evaluation_strategy": "steps",

    # Save checkpoint every 100 steps, keep best 3
    "save_steps": 100,
    "save_strategy": "steps",
    "save_total_limit": 3,

    # Load the checkpoint with lowest eval loss at the end
    "load_best_model_at_end": True,
    "metric_for_best_model": "eval_loss",

    # Optimizer: 8-bit Adam (saves ~2GB VRAM vs standard Adam32)
    # Unsloth provides a fused version that's slightly faster
    "optim": "adamw_8bit",

    # Weight decay: L2 regularization on weights (not biases)
    # Helps prevent overfitting on small datasets
    "weight_decay": 0.01,

    # Gradient clipping: prevents exploding gradients
    # If gradient norm exceeds 1.0, clip it to 1.0
    "max_grad_norm": 1.0,

    # WandB experiment tracking
    "report_to": "wandb",

    # Directory for checkpoints
    "output_dir": "./checkpoints",
}

Path(TRAIN_CONFIG["output_dir"]).mkdir(exist_ok=True)
print("✓ Configuration loaded")
print(f"  Base model: {MODEL_CONFIG['base_model']}")
print(f"  LoRA rank: {LORA_CONFIG['r']}")
print(f"  Effective batch size: {TRAIN_CONFIG['per_device_train_batch_size'] * TRAIN_CONFIG['gradient_accumulation_steps']}")
print(f"  Learning rate: {TRAIN_CONFIG['learning_rate']}")

/usr/local/lib/python3.12/dist-packages/unsloth/__init__.py:1531: UserWarning: WARNING: Unsloth should be imported before [trl, transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
✓ Configuration loaded
  Base model: Qwen/Qwen2.5-1.5B-Instruct
  LoRA rank: 16
  Effective batch size: 16
  Learning rate: 0.0002


In [5]:
from google.colab import drive
drive.mount('/content/drive')

DATASET_PATH = "/content/drive/MyDrive/fine-tuning-project/curated_data"
dataset = load_from_disk(DATASET_PATH)

print(f"Train:      {len(dataset['train']):,}")
print(f"Validation: {len(dataset['validation']):,}")
print(f"Test:       {len(dataset['test']):,}  ← sealed")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Train:      2,500
Validation: 2,732
Test:       2,732  ← sealed


In [6]:
# Cell 4: Verify your data is in the exact format SFTTrainer expects
# A format bug here will silently cause training to fail

def verify_example_format(example: dict) -> dict:
    """
    Check that an example has the correct ChatML structure.
    Returns a dict of what was found.
    """
    issues = []

    # Messages field must exist and be parseable
    if "messages" not in example:
        issues.append("Missing 'messages' field")
        return {"valid": False, "issues": issues}

    try:
        messages = json.loads(example["messages"]) if isinstance(example["messages"], str) else example["messages"]
    except:
        issues.append("'messages' field is not valid JSON")
        return {"valid": False, "issues": issues}

    # Must have exactly 3 messages
    if len(messages) != 3:
        issues.append(f"Expected 3 messages (system/user/assistant), got {len(messages)}")

    # Check roles
    expected_roles = ["system", "user", "assistant"]
    actual_roles = [m.get("role", "") for m in messages]
    if actual_roles != expected_roles:
        issues.append(f"Wrong role order: {actual_roles}, expected {expected_roles}")

    # Check assistant content is valid JSON
    if len(messages) == 3:
        assistant_content = messages[2].get("content", "")
        try:
            json.loads(assistant_content)
        except:
            issues.append(f"Assistant content is not valid JSON: {assistant_content[:100]}")

    return {
        "valid": len(issues) == 0,
        "issues": issues,
        "roles": actual_roles,
        "user_length": len(messages[1]["content"]) if len(messages) > 1 else 0,
        "assistant_length": len(messages[2]["content"]) if len(messages) == 3 else 0,
    }


# Verify 50 random examples
print("── Format Verification (50 random samples) ──\n")
valid_count = 0
invalid_examples = []

sample_indices = random.sample(range(len(dataset["train"])), 50)
for idx in sample_indices:
    ex = dataset["train"][idx]
    result = verify_example_format(ex)
    if result["valid"]:
        valid_count += 1
    else:
        invalid_examples.append((idx, result["issues"]))

print(f"Valid: {valid_count}/50")
if invalid_examples:
    print(f"\nInvalid examples found:")
    for idx, issues in invalid_examples[:5]:
        print(f"  Index {idx}: {issues}")
else:
    print("✓ All sampled examples are correctly formatted")

# Show one formatted example clearly
print("\n── Sample Training Example ──")
sample = dataset["train"][0]
messages = json.loads(sample["messages"]) if isinstance(sample["messages"], str) else sample["messages"]
for msg in messages:
    print(f"\n┌─ {msg['role'].upper()} ─")
    print(f"│  {msg['content'][:300]}{'...' if len(msg['content']) > 300 else ''}")
print("└─")

── Format Verification (50 random samples) ──

Valid: 50/50
✓ All sampled examples are correctly formatted

── Sample Training Example ──

┌─ SYSTEM ─
│  You are a precise function-calling assistant. Given a natural language request, extract the appropriate function call and return ONLY a valid JSON array containing the function call(s). Return no explanation, no markdown, no preamble — only the JSON array.

┌─ USER ─
│  Fetch details for the area with ID '123'.

┌─ ASSISTANT ─
│  [{"name": "areas_id", "arguments": {"is_id": 123}}]
└─


In [7]:
# Cell 5: Load the base model
# READ every comment here — these decisions affect your entire training run

print(f"Loading {MODEL_CONFIG['base_model']}...")
print(f"4-bit quantization: {MODEL_CONFIG['load_in_4bit']}")
print(f"Max sequence length: {MODEL_CONFIG['max_seq_length']}\n")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_CONFIG["base_model"],

    # Maximum sequence length this model will handle
    # VRAM scales with seq_len²: 2048 uses ~4x more attention memory than 1024
    max_seq_length=MODEL_CONFIG["max_seq_length"],

    # 4-bit NormalFloat quantization (NF4)
    # The frozen base model weights are stored in 4-bit instead of 16-bit
    # This reduces VRAM by ~4x for the frozen weights
    # LoRA adapters STILL train in full precision (bfloat16/float16)
    load_in_4bit=MODEL_CONFIG["load_in_4bit"],

    # Auto-detect: bfloat16 on A100, float16 on T4
    dtype=MODEL_CONFIG["dtype"],
)

print(f"\n✓ Model loaded successfully")
print(f"  Model type: {type(model).__name__}")
print(f"  Total parameters: {sum(p.numel() for p in model.parameters()):,}")

Loading Qwen/Qwen2.5-1.5B-Instruct...
4-bit quantization: True
Max sequence length: 2048

==((====))==  Unsloth 2026.8.18: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]


✓ Model loaded successfully
  Model type: Qwen2ForCausalLM
  Total parameters: 1,017,984,512


In [9]:
 # Cell 6: Examine what the model looks like after quantization
# THIS IS EDUCATIONAL — take 10 minutes to actually read the output

print("── Model Architecture Inspection ──\n")
print("First 3 transformer layers:")
print(model.model.layers[0])

print("\n── Weight Dtype Analysis ──")
print("What dtypes are the weights stored in?\n")

dtype_counts = defaultdict(int)
quantized_count = 0
full_precision_count = 0

for name, param in model.named_parameters():
    dtype_counts[str(param.dtype)] += 1
    if '4bit' in str(param.__class__.__name__).lower() or 'nf4' in str(param.dtype).lower():
        quantized_count += 1
    else:
        full_precision_count += 1

print("Dtype distribution:")
for dtype, count in sorted(dtype_counts.items()):
    print(f"  {dtype}: {count} tensors")

print(f"\nApproximate VRAM breakdown:")
# Estimate VRAM usage
total_params = sum(p.numel() for p in model.parameters())
vram_if_fp16 = total_params * 2 / 1e9   # 2 bytes per param in fp16
vram_if_4bit = total_params * 0.5 / 1e9  # 0.5 bytes per param in 4-bit
print(f"  If loaded in fp16: ~{vram_if_fp16:.1f} GB")
print(f"  Loaded in 4-bit:   ~{vram_if_4bit:.1f} GB")
print(f"  Actual VRAM used:  ~{torch.cuda.memory_allocated() / 1e9:.1f} GB")
print(f"  VRAM saved:        ~{vram_if_fp16 - torch.cuda.memory_allocated() / 1e9:.1f} GB")

print("""
── What This Means (write this in your notebook) ──

The base model weights are stored in 4-bit NormalFloat (NF4) format.
This is NOT how you trained in high school math — these aren't just rounded integers.
NF4 is a data type optimized for the distribution of neural network weights,
which tend to follow a normal (Gaussian) distribution.

The weights can ONLY be used for the forward pass in 4-bit.
When you add LoRA and train, the optimizer updates happen in float32.
The LoRA adapters themselves (A and B matrices) live in float32/bfloat16.

So QLoRA = 4-bit frozen base + full-precision LoRA adapters
""")


── Model Architecture Inspection ──

First 3 transformer layers:
Qwen2DecoderLayer(
  (self_attn): Qwen2Attention(
    (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
    (k_proj): Linear(in_features=1536, out_features=256, bias=True)
    (v_proj): Linear(in_features=1536, out_features=256, bias=True)
    (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
    (rotary_emb): LlamaRotaryEmbedding()
  )
  (mlp): Qwen2MLP(
    (gate_proj): Linear4bit(in_features=1536, out_features=8960, bias=False)
    (up_proj): Linear4bit(in_features=1536, out_features=8960, bias=False)
    (down_proj): Linear4bit(in_features=8960, out_features=1536, bias=False)
    (act_fn): SiLUActivation()
  )
  (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
  (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
)

── Weight Dtype Analysis ──
What dtypes are the weights stored in?

Dtype distribution:
  torch.float16: 168 tensors
  torch.uint8: 170 tensors

Approximate VRAM 

In [10]:
# Cell 7: Add LoRA adapters to the model
# Read every comment. Know what each line does before running.

print("── LoRA Configuration Explanation ──\n")
print("""
WHAT IS LORA?
─────────────
Instead of updating the full weight matrix W (shape: d_model × d_model),
LoRA adds two small matrices:

    W_new = W_frozen + B × A

    where:
      W_frozen: original weight, frozen, 4-bit
      A: shape (d_model × r) — the "down" projection, initialized randomly
      B: shape (r × d_model) — the "up" projection, initialized to zeros
      r: rank (your hyperparameter)

WHY DOES B START AT ZERO?
─────────────────────────
At initialization: B × A = 0 × A = 0
So W_new = W_frozen + 0 = W_frozen

This means at the start of training, LoRA adds nothing.
Training starts from exactly the base model — no random disruption.
Gradients flow through B first (since B × A passes through B before A in backward),
then A adjusts to create useful low-rank updates.

WHAT DOES RANK r CONTROL?
──────────────────────────
r = 4:  ~0.13% of params updated. Very efficient, may underfit complex tasks.
r = 8:  ~0.25% of params. Good for simple tasks.
r = 16: ~0.5% of params. Standard starting point.
r = 32: ~1.0% of params. More capacity, more risk of overfitting small datasets.
r = 64: ~2.0% of params. Usually overkill for 2500 examples.

For your 2500-example JSON extraction task → start with r=16.

WHAT IS lora_alpha?
───────────────────
alpha controls the effective learning rate of the LoRA update:
    scaled_update = (alpha / r) × (B × A)

With r=16 and alpha=32: scale = 32/16 = 2.0
This means LoRA updates are amplified 2x relative to the base weights.

The heuristic: alpha = 2r keeps scale constant regardless of r.
Some practitioners prefer alpha = r (scale = 1.0) for conservative fine-tuning.

WHY TARGET THESE MODULES?
──────────────────────────
q_proj, k_proj, v_proj, o_proj: The 4 attention projections.
    These handle WHAT the model pays attention to.
    Updating these teaches the model to focus on JSON-relevant parts of input.

gate_proj, up_proj, down_proj: The 3 MLP projections (SwiGLU architecture).
    These handle HOW the model transforms information.
    Updating these teaches the model the specific transformations for JSON output.

Targeting all 7 is aggressive but appropriate for a format-heavy task.
Start here; if you see overfitting, try just q_proj + v_proj.
""")

# Apply LoRA to the model
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_CONFIG["r"],
    target_modules=LORA_CONFIG["target_modules"],
    lora_alpha=LORA_CONFIG["lora_alpha"],
    lora_dropout=LORA_CONFIG["lora_dropout"],
    bias=LORA_CONFIG["bias"],
    use_gradient_checkpointing=LORA_CONFIG["use_gradient_checkpointing"],
    random_state=LORA_CONFIG["random_state"],
    use_rslora=LORA_CONFIG["use_rslora"],
)

print("✓ LoRA adapters added")

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.


── LoRA Configuration Explanation ──


WHAT IS LORA?
─────────────
Instead of updating the full weight matrix W (shape: d_model × d_model),
LoRA adds two small matrices:

    W_new = W_frozen + B × A
    
    where:
      W_frozen: original weight, frozen, 4-bit
      A: shape (d_model × r) — the "down" projection, initialized randomly
      B: shape (r × d_model) — the "up" projection, initialized to zeros
      r: rank (your hyperparameter)

WHY DOES B START AT ZERO?
─────────────────────────
At initialization: B × A = 0 × A = 0
So W_new = W_frozen + 0 = W_frozen

This means at the start of training, LoRA adds nothing.
Training starts from exactly the base model — no random disruption.
Gradients flow through B first (since B × A passes through B before A in backward),
then A adjusts to create useful low-rank updates.

WHAT DOES RANK r CONTROL?
──────────────────────────
r = 4:  ~0.13% of params updated. Very efficient, may underfit complex tasks.
r = 8:  ~0.25% of params. Good for si

Unsloth 2026.8.18 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


✓ LoRA adapters added
